In [ ]:
import torch
from PIL import Image
from lang_sam import LangSAM
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from rasterio.windows import Window
from huggingface_hub import snapshot_download
import os


In [ ]:
from lang_sam import LangSAM
CACHE_DIR = "./local_langsam"
sam_ckpt = f"{CACHE_DIR}/sam2.1_hiera_large.pt"
gdino_ckpt = f"{CACHE_DIR}/groundingdino_hf_model"
gdino_proc = f"{CACHE_DIR}/bert-base-uncased"

model = LangSAM(
    sam_type="sam2.1_hiera_large",
    sam_ckpt_path=sam_ckpt,
    gdino_model_ckpt_path=gdino_ckpt,
    gdino_processor_ckpt_path=gdino_proc
)

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from PIL import Image

def get_full_mask_fast(model, image_bytes, keyword="tree", patch_size=1024, overlap=128, text_threshold=0.15):
    stride = patch_size - overlap
    with rasterio.MemoryFile(image_bytes) as memfile, memfile.open() as src:
        h, w = src.height, src.width
        full_mask = np.zeros((h, w), dtype=bool)
        
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                th, tw = min(patch_size, h - y), min(patch_size, w - x)
                
                # read and format patch
                patch = src.read([1, 2, 3], window=Window(x, y, tw, th))
                patch = np.moveaxis(patch, 0, -1)
                
                if patch.dtype == np.uint16:
                    patch = (patch >> 8).astype(np.uint8) # same as patch / 256
                
                try:
                    # Predict and extract safely using .get()
                    res = model.predict([Image.fromarray(patch)], [keyword], text_threshold=text_threshold)
                    masks = res[0].get('masks') if res else None
                    
                    if masks is not None and len(masks) > 0:
                        # tensor conversion
                        m_np = masks.cpu().numpy().copy() if hasattr(masks, 'cpu') else masks
                        c_mask = np.any(m_np, axis=0) if m_np.ndim == 3 else m_np
                        
                        # Resize if necessary for edges
                        if c_mask.shape != (th, tw):
                            c_mask = np.array(Image.fromarray(c_mask).resize((tw, th), Image.NEAREST))
                        
                        full_mask[y:y+th, x:x+tw] |= c_mask.astype(bool)
                        
                except Exception as e:
                    print(f"skipping tile at {x},{y}: {e}") 

        return full_mask.astype(np.uint8) * 255 
            

In [ ]:
image_path = "/home/ubuntu/work/satellite_data/digital_orthophoto_nrw/2021/2021/dop10rgbi_32_280_5652_1_nw_2021.tif"
with open(image_path, "rb") as f:
    img_bytes = f.read()
    
masks = get_full_mask_fast(
    model = model,
    image_bytes = img_bytes,
    keyword="tree",
    patch_size=1024,
    overlap=0,
    text_threshold=0.15
)

In [ ]:
import rasterio
from rasterio.plot import show

# Use the same path variable
with rasterio.open(image_path) as src:
    show(src)

In [ ]:
def plot_overview_and_roi(image_path, full_mask_array, roi_x=2000, roi_y=3000, roi_size=1024, scale_factor=0.2):
    with rasterio.open(image_path) as src:
        # Read the image (bands 1, 2, 3)
        img_data = src.read([1, 2, 3])
        # Transpose from (C, H, W) to (H, W, C) for matplotlib
        img_data = np.transpose(img_data, (1, 2, 0))
        # No normalization needed since already uint8

    # DEBUG: Check mask values
    print(f"Mask unique values: {np.unique(full_mask_array)}")
    print(f"Mask dtype: {full_mask_array.dtype}")
    print(f"Mask min: {full_mask_array.min()}, max: {full_mask_array.max()}")

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Overview
    overview_img = img_data[::int(1/scale_factor), ::int(1/scale_factor)]
    mask_small = full_mask_array[::int(1/scale_factor), ::int(1/scale_factor)]
    
    
    overlay_overview = overview_img.copy()
    overlay_overview[mask_small > 0] = [0, 255, 0]  # Green for any positive detection
    overview_result = (overview_img * 0.7 + overlay_overview * 0.3).astype(np.uint8)
    
    axes[0].imshow(overview_result)
    axes[0].axis('off')
    axes[0].set_title(f'Overview (scale factor: {scale_factor})')
    
    # ROI
    roi_img = img_data[roi_y:roi_y + roi_size, roi_x:roi_x + roi_size]
    roi_mask = full_mask_array[roi_y:roi_y + roi_size, roi_x:roi_x + roi_size]
    
    # Create overlay for ROI - use > 0
    overlay_roi = roi_img.copy()
    overlay_roi[roi_mask > 0] = [0, 255, 0]  # Green for any positive detection
    
    roi_result = (roi_img * 0.6 + overlay_roi * 0.4).astype(np.uint8)
    
    axes[1].imshow(roi_result)
    axes[1].axis('off')
    axes[1].set_title(f'ROI (size: {roi_size}x{roi_size})')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_overview_and_roi(image_path, masks, roi_x=1500, roi_y=3000, roi_size=1000, scale_factor=0.2) # 10%

In [ ]:
masks_solar = get_full_mask_fast(
    model = model,
    image_bytes = img_bytes,
    keyword="house",
    patch_size=1024,
    overlap=128,
    text_threshold=0.15
)

In [ ]:

plot_overview_and_roi(image_path, masks_solar, roi_x=2000, roi_y=2000, roi_size=1000, scale_factor=0.2)